# Качество погодных данных

Отдельно проверяем четыре числовых кандидата на всем CSV.
Считаем подозрительные значения и их связь с меткой, без автоматической очистки.
Настройка путей и ролей ниже позволяет выполнить этот ноутбук самостоятельно.


In [1]:
# Используем пути относительно проекта, без привязки к компьютеру
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

data_path = project_root / 'data' / 'raw' / 'go_arounds_augmented.csv.gz'
dictionary_path = project_root / 'docs' / 'data_dictionary.md'
if not data_path.is_file() or not dictionary_path.is_file():
    raise FileNotFoundError('Нужны исходный CSV и docs/data_dictionary.md')

print('pandas:', pd.__version__)
print('Датасет:', data_path.name)


pandas: 3.0.6
Датасет: go_arounds_augmented.csv.gz


In [2]:
# Роль поля задаем явно: так легче проверить, что случайно не попало в модель
target_column = 'has_ga'
time_column = 'time'
identifier_columns = ['icao24', 'callsign', 'registration']
excluded_columns = ['runway', 'n_approaches', 'n_rwy_approached']

categorical_features = ['airport', 'typecode', 'wtc']
numeric_features = [
    'wind_speed_knts', 'visibility_m', 'temperature_deg', 'press_p',
]
candidate_features = categorical_features + numeric_features
deferred_columns = [
    'icaoaircrafttype', 'glide_slope_angle', 'has_intersection', 'rwy_length',
    'airport_country', 'airport_region', 'operator_country', 'operator_region',
    'wind_dir_deg', 'wind_gust_knts', 'press_sea_level_p',
    'weather_intensity', 'weather_precipitation', 'weather_desc',
    'weather_obscuration', 'weather_other',
]

# Проверяем только заголовок: для контроля схемы не нужна полная загрузка
columns = pd.read_csv(data_path, nrows=0).columns.tolist()
assigned_columns = (
    [target_column, time_column] + identifier_columns + excluded_columns
    + candidate_features + deferred_columns
)
if len(assigned_columns) != len(set(assigned_columns)):
    raise ValueError('Одно поле получило несколько ролей')
if set(columns) != set(assigned_columns):
    raise ValueError('Схема CSV отличается от словаря ролей')

documented_columns = [
    line.split('|')[1].strip().strip('`')
    for line in dictionary_path.read_text(encoding='utf-8').splitlines()
    if line.startswith('| `')
]
if documented_columns != columns:
    raise ValueError('Словарь не соответствует порядку полей исходного CSV')

print('Все поля получили роль:', len(columns))
print('Категориальные кандидаты:', categorical_features)
print('Числовые кандидаты:', numeric_features)


Все поля получили роль: 31
Категориальные кандидаты: ['airport', 'typecode', 'wtc']
Числовые кандидаты: ['wind_speed_knts', 'visibility_m', 'temperature_deg', 'press_p']


## Давление: ограничения интерпретации

`press_p` заявлено авторами как QNH в гПа. Нельзя автоматически трактовать
отклонения как QFE, ошибки или смешение единиц. Вид давления, единицы и
преобразование проверим по исходному METAR, территории и аэродрому.
`press_sea_level_p` не подставляем вместо QNH. Отдельное поле QFE отсутствует.

Граница 850–1100 – только диагностический фильтр. Найденные записи остаются
без изменений. План проверки: [словарь данных](../docs/data_dictionary.md).


## Проверка подозрительных погодных значений

В первом аудите найдены экстремальные максимумы. Ниже считаем записи вне
широких рабочих диапазонов: ветер 0–150 узлов, температура от −80 до 60 °C,
`press_p` 850–1100 гПа и видимость 0–100 000 м. Границы – наши диагностические
правила, а не стандарт METAR или готовые правила удаления строк.
NaN учитываем отдельно. Значения не исправляем и строки не удаляем.

Читаем CSV частями, чтобы не держать весь набор в памяти. Одновременно
считаем число строк и уходов по месяцам для обсуждения временного разбиения.


In [4]:
# Диапазоны задаем явно и используем пока только для диагностики
audit_limits = {
    'wind_speed_knts': (0, 150),
    'visibility_m': (0, 100_000),
    'temperature_deg': (-80, 60),
    'press_p': (850, 1100),
}
audit_counts = {
    column: {'missing': 0, 'outside_limits': 0, 'ga_outside_limits': 0}
    for column in numeric_features
}
audit_examples = {column: [] for column in numeric_features}
month_parts = []
total_rows = 0
invalid_times = 0
invalid_targets = 0

# Число строк в части ограничивает память, но не объем проверяемых данных
read_columns = [time_column, target_column] + numeric_features
for chunk in pd.read_csv(data_path, usecols=read_columns, chunksize=250_000):
    total_rows += len(chunk)
    chunk_times = pd.to_datetime(chunk[time_column], errors='coerce', utc=True)
    invalid_times += int(chunk_times.isna().sum())
    invalid_targets += int((~chunk[target_column].isin([True, False])).sum())

    for column, (lower, upper) in audit_limits.items():
        values = chunk[column]
        outside = values.notna() & (values.lt(lower) | values.gt(upper))
        audit_counts[column]['missing'] += int(values.isna().sum())
        audit_counts[column]['outside_limits'] += int(outside.sum())
        audit_counts[column]['ga_outside_limits'] += int(
            (outside & chunk[target_column].eq(True)).sum()
        )
        if len(audit_examples[column]) < 5:
            examples = values[outside].drop_duplicates().tolist()
            audit_examples[column] = list(dict.fromkeys(
                audit_examples[column] + examples
            ))[:5]

    monthly = chunk.assign(month=chunk_times.dt.strftime('%Y-%m')).groupby('month').agg(
        rows=(target_column, 'size'), go_arounds=(target_column, 'sum'),
    )
    month_parts.append(monthly)

audit_summary = pd.DataFrame.from_dict(audit_counts, orient='index')
audit_summary['outside_percent'] = (
    audit_summary['outside_limits'] / total_rows * 100
).round(4)
audit_summary['examples'] = pd.Series(audit_examples)
monthly_summary = pd.concat(month_parts).groupby(level=0).sum().sort_index()
monthly_summary['ga_percent'] = (
    monthly_summary['go_arounds'] / monthly_summary['rows'] * 100
).round(4)

print('Проверено строк:', total_rows)
print('Нераспознанных дат:', invalid_times)
print('Недопустимых значений метки:', invalid_targets)
display(audit_summary)
display(monthly_summary)


Проверено строк: 8969145
Нераспознанных дат: 0
Недопустимых значений метки: 0


                 missing  outside_limits  ga_outside_limits  outside_percent                                                    examples
wind_speed_knts     6846               7                  0           0.0001                                         [194.384, 622.0288]
visibility_m        1123               0                  0           0.0000                                                          []
temperature_deg      982             344                  0           0.0038                         [2018.0, 311.0, 225.0, 322.0, 80.0]
press_p             1394           42561                 96           0.4745  [3386.051361, 800.0, 1354.556, 1693.1950000000002, 2000.0]

           rows  go_arounds  ga_percent
month                                  
2019-01  634696        2577      0.4060
2019-02  616874        2733      0.4430
2019-03  656990        2696      0.4104
2019-04  673813        2434      0.3612
2019-05  717947        2578      0.3591
2019-06  727833        2719      0.3736
2019-07  807313        2770      0.3431
2019-08  829358        2945      0.3551
2019-09  772287        2641      0.3420
2019-10  876435        3338      0.3809
2019-11  818428        3025      0.3696
2019-12  837171        3208      0.3832

## Вывод по проверке качества

Проверены все 8 969 145 строк. Ошибок разбора дат и недопустимых значений
метки нет. Диапазоны ниже – наши диагностические границы, не стандарт METAR.

| Поле | Рабочая граница | Вне границ | Среди них has_ga=True |
| --- | --- | --- | --- |
| wind_speed_knts | 0–150 узлов | 7 | 0 |
| temperature_deg | от −80 до 60 °C | 344 | 0 |
| press_p | 850–1100 гПа | 42 561 | 96 |
| visibility_m | 0–100 000 м | 0 | 0 |

Примеры: ветер 194,384 и 622,0288 узла; температура 2018, 311, 225 и 322 °C;
`press_p` 3386,05, 800, 1354,56, 1693,20 и 2000 гПа. Это первые найденные
уникальные значения, не случайная выборка. Частоты по полям нельзя складывать:
одна строка может нарушать несколько правил. NaN учитываются отдельно.

Данные не изменены. Причины отклонений не установлены. Нулевая частота
нарушений диапазона не доказывает качество поля. Остальные поля подробно
на аномалии здесь не проверены.

Особое внимание нужно давлению: проверка затрагивает 96 записей с уходом.
До обработки изучим контекст этих строк и влияние решения на редкий класс.
Таблица по месяцам подготовлена для обсуждения временного разбиения.

Решение об очистке принимает автор проекта после разбора примеров.
Возможные подходы: исключение поля, маркировка подозрительных значений как
пропусков либо исправление после подтверждения причины. Ни один подход
пока не применен. Разбиение и обучение не выполнены.
